# Translate OpenMath → Indonesian JSONL
Output schema: `soal`, `cara`, `jawaban`

In [ ]:
# ── CONFIGURE: fill INPUT_JSON after you upload the dataset ─────────────────
INPUT_JSON   = "/kaggle/input/REPLACE_WITH_YOUR_DATASET_SLUG/openmath_reasoning_20k.json"
OUTPUT_JSONL = "/kaggle/working/openmath_reasoning_indo.jsonl"
BATCH_SIZE   = 256   # T4/P100 16GB → safe at 256
# ───────────────────────────────────────────────────────────────────────────

In [ ]:
!pip install -q sentencepiece sacremoses

In [ ]:
import json, re
from pathlib import Path
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from tqdm.notebook import tqdm

TRANSLATION_MODEL = "Helsinki-NLP/opus-mt-en-id"
MAX_SRC_TOKENS = 480

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
def strip_think(text: str) -> str:
    idx = text.find("</think>")
    return text[idx + 8:].strip() if idx >= 0 else text.strip()

def protect_latex(text: str):
    placeholders = {}
    counter = [0]
    def replace(m):
        key = f"__M{counter[0]}__"
        placeholders[key] = m.group(0)
        counter[0] += 1
        return key
    protected = re.sub(r"\\\(.*?\\\)|\$[^$]+\$", replace, text, flags=re.DOTALL)
    return protected, placeholders

def restore_latex(text: str, placeholders: dict) -> str:
    for key, val in placeholders.items():
        text = text.replace(key, val)
    return text

def split_by_words(text: str, tokenizer, max_tokens: int):
    words = text.split()
    chunks, current = [], []
    for word in words:
        candidate = " ".join(current + [word])
        if len(tokenizer.encode(candidate, add_special_tokens=False)) > max_tokens and current:
            chunks.append(" ".join(current))
            current = [word]
        else:
            current.append(word)
    if current:
        chunks.append(" ".join(current))
    return chunks or [""]

def chunk_text(text: str, tokenizer, max_tokens: int):
    paragraphs = text.split("\n")
    chunks, current = [], ""
    for para in paragraphs:
        if len(tokenizer.encode(para, add_special_tokens=False)) > max_tokens:
            if current:
                chunks.append(current)
                current = ""
            chunks.extend(split_by_words(para, tokenizer, max_tokens))
            continue
        candidate = (current + "\n" + para).strip() if current else para
        if len(tokenizer.encode(candidate, add_special_tokens=False)) > max_tokens and current:
            chunks.append(current)
            current = para
        else:
            current = candidate
    if current:
        chunks.append(current)
    return chunks or [""]

def translate_batch(texts, tokenizer, model, device):
    inputs = tokenizer(
        texts, return_tensors="pt", padding=True,
        truncation=True, max_length=MAX_SRC_TOKENS,
    ).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=512)
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

print("Helpers defined.")

In [ ]:
print(f"Reading {INPUT_JSON} ...")
with open(INPUT_JSON, encoding="utf-8") as f:
    data = json.load(f)
print(f"Loaded {len(data):,} entries")
print("Keys:", list(data[0].keys()))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading {TRANSLATION_MODEL} on {device} ...")
tokenizer = AutoTokenizer.from_pretrained(TRANSLATION_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(TRANSLATION_MODEL).to(device)
model.eval()
print("Model ready.")

In [ ]:
FIELDS = ["problem", "generated_solution", "expected_answer"]

tasks = []
for i, entry in enumerate(data):
    for field in FIELDS:
        raw = entry.get(field, "") or ""
        if field == "generated_solution":
            raw = strip_think(raw)
        if not raw.strip():
            continue
        protected, ph = protect_latex(raw)
        chunks = chunk_text(protected, tokenizer, MAX_SRC_TOKENS)
        tasks.append((i, field, chunks, ph))

flat = [
    (t_idx, c_idx, chunk)
    for t_idx, task in enumerate(tasks)
    for c_idx, chunk in enumerate(task[2])
]
print(f"Tasks: {len(tasks):,}  |  Chunks: {len(flat):,}  |  Batches @{BATCH_SIZE}: {-(-len(flat)//BATCH_SIZE):,}")

In [ ]:
chunk_results = {}
for start in tqdm(range(0, len(flat), BATCH_SIZE), desc="Translating"):
    batch = flat[start : start + BATCH_SIZE]
    texts = [b[2] for b in batch]
    translated = translate_batch(texts, tokenizer, model, device)
    for (t_idx, c_idx, _), result in zip(batch, translated):
        chunk_results[(t_idx, c_idx)] = result

print("Translation done.")

In [ ]:
translations = {}
for t_idx, (i, field, chunks, ph) in enumerate(tasks):
    parts = [chunk_results.get((t_idx, c_idx), "") for c_idx in range(len(chunks))]
    translations[(i, field)] = restore_latex("\n".join(parts), ph)

records = []
for i, entry in enumerate(data):
    records.append({
        "soal":    translations.get((i, "problem"),            entry.get("problem", "")),
        "cara":    translations.get((i, "generated_solution"), strip_think(entry.get("generated_solution", ""))),
        "jawaban": translations.get((i, "expected_answer"),    entry.get("expected_answer", "")),
    })

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

size_mb = Path(OUTPUT_JSONL).stat().st_size / 1e6
print(f"Written {len(records):,} entries → {OUTPUT_JSONL} ({size_mb:.1f} MB)")
print("\nSample row:")
print(json.dumps(records[0], ensure_ascii=False, indent=2)[:600])